In [0]:

# Create dropdown widget with file options
dbutils.widgets.dropdown(
  name="file_name",
  defaultValue="amit_donations.txt",
  choices=["amit_donations.txt", "mark_donations.txt"],
  label="Select File to Process"
)



In [0]:
# Get selected file from widget
selected_file = dbutils.widgets.get("file_name")

# Define paths
volume_path = f"/Volumes/amit/default/donations/{selected_file}"
bronze_table = "amit.default.donations_bronze"
prisoner_name = selected_file.split('_')[0]
print(f"Selected file: {selected_file}")
print(f"Source path: {volume_path}")
print(f"Target table: {bronze_table}")



In [0]:
from pyspark.sql.functions import current_timestamp, lit, row_number, col, when, concat_ws
from pyspark.sql.window import Window

# Read the text file
df = spark.read.text(volume_path)

# Filter out empty lines
df_clean = df.filter("value IS NOT NULL AND TRIM(value) != ''")

# Add line numbers
window_spec = Window.orderBy(lit(1))
df_numbered = df_clean.withColumn("line_num", row_number().over(window_spec))

# Identify donation start lines (lines that are just numbers)
df_with_donation_id = df_numbered.withColumn(
  "is_donation_start",
  col("value").rlike("^[0-9]+$")
)

# Create a temporary view for SQL processing
df_with_donation_id.createOrReplaceTempView("numbered_lines")

# Use SQL to group lines into donations
# Key insight: last line of each donation is the amount (contains ₪)
donations_df = spark.sql("""
  WITH donation_starts AS (
    SELECT 
      line_num,
      value as donation_id
    FROM numbered_lines
    WHERE is_donation_start = true
  ),
  lines_with_donation AS (
    SELECT 
      nl.line_num,
      nl.value,
      ds.donation_id,
      nl.line_num - ds.line_num as offset_in_donation
    FROM numbered_lines nl
    LEFT JOIN donation_starts ds 
      ON nl.line_num >= ds.line_num
    QUALIFY ROW_NUMBER() OVER (PARTITION BY nl.line_num ORDER BY ds.line_num DESC) = 1
  ),
  donation_structure AS (
    SELECT
      donation_id,
      MAX(CASE WHEN offset_in_donation = 1 THEN value END) as donor_name,
      MAX(CASE WHEN offset_in_donation = 2 THEN value END) as time_text,
      MAX(CASE WHEN value LIKE '%₪%' THEN value END) as amount_text,
      CONCAT_WS('\\n', 
        COLLECT_LIST(
          CASE 
            WHEN offset_in_donation > 2 AND value NOT LIKE '%₪%' 
            THEN value 
          END
        )
      ) as comment_text
    FROM lines_with_donation
    WHERE donation_id IS NOT NULL
    GROUP BY donation_id
  )
  SELECT
    donation_id,
    donor_name,
    time_text,
    amount_text,
    CASE WHEN comment_text = '' THEN NULL ELSE comment_text END as comment_text
  FROM donation_structure
""")

# Add metadata columns
df_with_metadata = donations_df \
  .withColumn("source_file", lit(selected_file)) \
  .withColumn("prisoner", lit(prisoner_name)) \
  .withColumn("ingestion_timestamp", current_timestamp())

# Write to bronze table (append mode)
df_with_metadata.write \
  .mode("append") \
  .saveAsTable(bronze_table)

print(f"✓ Successfully loaded {donations_df.count()} donations from {selected_file} to {bronze_table}")
print(f"  Prisoner: {prisoner_name}")

In [0]:
# Display the latest records from bronze table
display(
  spark.table(bronze_table)
    .filter(f"source_file = '{selected_file}'")
    .orderBy("ingestion_timestamp", ascending=False)
    .limit(20)
)